In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers

In [ ]:
!pip install -q -U sentence-transformers faiss-cpu transformers torch

In [8]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ============================================================
# 1. KNOWLEDGE BASE
# ============================================================

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",

    "Retrieval-Augmented Generation combines document retrieval "
    "with text generation.",

    "Python is a popular high-level programming language "
    "used in AI development.",

    "Vector databases store embeddings and support fast "
    "similarity search."
]


# ============================================================
# 2. LOAD EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")


# ============================================================
# 3. CONVERT DOCUMENTS INTO EMBEDDINGS
# ============================================================

doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
)

print("Document embeddings created.")
print("Embedding shape:", doc_embeddings.shape)


# ============================================================
# 4. CREATE FAISS VECTOR INDEX
# ============================================================

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

# FAISS expects float32
doc_embeddings = doc_embeddings.astype("float32")

index.add(doc_embeddings)

print("FAISS index created.")
print("Number of documents:", index.ntotal)


# ============================================================
# 5. USER QUERY
# ============================================================

query = "What is RAG in AI?"

print("\nUser Query:")
print(query)


# ============================================================
# 6. CONVERT QUERY INTO EMBEDDING
# ============================================================

query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
)

query_embedding = query_embedding.astype("float32")


# ============================================================
# 7. SEARCH MOST RELEVANT DOCUMENTS
# ============================================================

k = 2

distances, indices = index.search(
    query_embedding,
    k
)


# ============================================================
# 8. RETRIEVE DOCUMENTS
# ============================================================

retrieved_chunks = []

for i in indices[0]:
    retrieved_chunks.append(documents[i])


print("\nRetrieved Documents:")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}")


# ============================================================
# 9. CREATE CONTEXT
# ============================================================

context = "\n".join(retrieved_chunks)


# ============================================================
# 10. CREATE AUGMENTED PROMPT
# ============================================================

prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""


print("\nPrompt sent to LLM:")
print(prompt)


# ============================================================
# 11. LOAD FLAN-T5 MODEL
# ============================================================

print("Loading FLAN-T5 model...")

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 model loaded.")


# ============================================================
# 12. TOKENIZE PROMPT
# ============================================================

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)


# ============================================================
# 13. GENERATE ANSWER
# ============================================================

outputs = model.generate(
    **inputs,
    max_new_tokens=60
)


# ============================================================
# 14. DECODE ANSWER
# ============================================================

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


# ============================================================
# 15. FINAL OUTPUT
# ============================================================

print("\n========================================")
print("FINAL ANSWER")
print("========================================")

print(answer)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Document embeddings created.
Embedding shape: (4, 384)
FAISS index created.
Number of documents: 4

User Query:
What is RAG in AI?

Retrieved Documents:
1. Python is a popular high-level programming language used in AI development.
2. Retrieval-Augmented Generation combines document retrieval with text generation.

Prompt sent to LLM:

Use the following context to answer the question.

Context:
Python is a popular high-level programming language used in AI development.
Retrieval-Augmented Generation combines document retrieval with text generation.

Question:
What is RAG in AI?

Answer:

Loading FLAN-T5 model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded.

FINAL ANSWER
(iii).
